## Running multiple circuits at once with CircuitBinding

This notebooks aims at presenting our equivalent in MPQP to qiskit's pubs and braket's ProgramSets.

The goal of this feature is to be able to send multiple circuits in one job and being able to retrieved the correct result with the run. This has the desired effect of reducing costs of sending circuits to providers and in certain cases get faster runs.

The term binding comes from the fact that you can "bind" a circuit with measurements (for example observables) and variables (in case of parametrized circuits) so that you can run the circuit(s) with different observables.

In [ ]:
# First we setup here the circuits, values and observables to be used
from mpqp.core.circuit import BindingMode, CircuitBinding
from mpqp.execution.devices import (
    AvailableDevice,
    IBMDevice,
    AWSDevice,
)

from mpqp import (
    CNOT,
    BasisMeasure,
    ExpectationMeasure,
    H,
    IBMDevice,
    Language,
    Observable,
    QCircuit,
    U,
    pI,
    pZ,
    pX,
    run,
)
from sympy import Symbol
from mpqp.execution.runner import run
from mpqp.execution.devices import IBMDevice, AWSDevice
import numpy as np

theta, phi, psi = Symbol('θ'), Symbol('phi'), Symbol('psi')
c1 = QCircuit([U(theta, phi, psi, 0)], label="c1")
c2 = QCircuit([H(0)], label="c2")
c3 = QCircuit([H(0), CNOT(0, 1)], label="c3")
c4 = QCircuit([H(0), H(1), CNOT(0, 1)], label="c4")

v1 = {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
v2 = {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
v3 = {'θ': 3.0, 'phi': 3.0, 'psi': 3.0}
v4 = {'θ': 4.0, 'phi': 4.0, 'psi': 4.0}
v_t0 = {'θ': 0}
v_t1 = {'θ': np.pi}

m1 = ExpectationMeasure(Observable(pI), label="Exp1", shots=2024)
m1_bis = ExpectationMeasure(Observable(pZ), label="Exp1bis", shots=2024)
m2 = ExpectationMeasure(Observable(pX @ pZ), label="Exp2", shots=2024)
m2_bis = ExpectationMeasure(Observable(pZ @ pZ), label="Exp2bis", shots=2024)
m3 = BasisMeasure(label="b3", shots=2024)
m3_bis = BasisMeasure(label="b3_bis", shots=1024)
m4 = None

m_I = ExpectationMeasure(Observable(pI), label="Exp_I", shots=2024)
m_Z = ExpectationMeasure(Observable(pZ), label="Exp_Z", shots=2024)

### Usecase 1: Running a list of circuit

The most basic case is the one where we just want to run circuits without any binding. In this case any `BasisMeasure` or `ExpectationMeasure` are embedded in the circuits and not provided in the `CircuitBinding` object.

In [ ]:
# Example 1: Two state vector jobs
# Note: state vector jobs are not supported by braket's ProgramSets
binding = CircuitBinding([c3, c4])
print(run(binding, IBMDevice.AER_SIMULATOR))

BatchResult: 2 results
    Result: c3, IBMDevice, AER_SIMULATOR
      State vector: [0.70711, 0, 0, 0.70711]
      Probabilities: [0.5, 0, 0, 0.5]
      Number of qubits: 2
    

    Result: c4, IBMDevice, AER_SIMULATOR
      State vector: [0.5, 0.5, 0.5, 0.5]
      Probabilities: [0.25, 0.25, 0.25, 0.25]
      Number of qubits: 2
    



In [ ]:
binding_with_basismeasure = CircuitBinding([c3 + QCircuit([m3]), c4 + QCircuit([m3])])
# or you can do
binding_with_basismeasure2 = CircuitBinding(
    [c3, c4], measurements=[m3], mode=BindingMode.PRODUCT
)
print(run(binding_with_basismeasure, AWSDevice.BRAKET_LOCAL_SIMULATOR))

0
BatchResult: 2 results
    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Counts: [1027, 0, 0, 997]
      Probabilities: [0.50741, 0, 0, 0.49259]
      Samples:
        State: 00, Index: 0, Count: 1027, Probability: 0.5074111
        State: 11, Index: 3, Count: 997, Probability: 0.4925889
      Error: None
    

    Result: c4, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Counts: [501, 538, 471, 514]
      Probabilities: [0.24753, 0.26581, 0.23271, 0.25395]
      Samples:
        State: 00, Index: 0, Count: 501, Probability: 0.2475296
        State: 01, Index: 1, Count: 538, Probability: 0.2658103
        State: 10, Index: 2, Count: 471, Probability: 0.2327075
        State: 11, Index: 3, Count: 514, Probability: 0.2539526
      Error: None
    



In [ ]:
binding_with_observables = CircuitBinding([c3 + QCircuit([m2]), c4 + QCircuit([m2])])

print(run(binding_with_observables, IBMDevice.AER_SIMULATOR))
print(run(binding_with_observables, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 2 results
    Result: c3, IBMDevice, AER_SIMULATOR
      Exp2_0:
    Expectation value: 0.014320987654320988
    Error/Variance: 0.02221994331995532
    

    Result: c4, IBMDevice, AER_SIMULATOR
      Exp2_0:
    Expectation value: -0.0004938271604938272
    Error/Variance: 0.022222219512608006
    

0
BatchResult: 2 results
    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Expectation value: 0.024703557312252964
      Error/Variance: None
    With observables: [pX@pZ]
    

    Result: c4, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Expectation value: -0.03557312252964427
      Error/Variance: None
    With observables: [pX@pZ]
    



### Usecase 2: Binding modes
#### BindingMode.PRODUCT:
For the first mode there is the product which is simply the products between the circuits, measurements and parameters.  
If we have `CircuitBinding([c1,c2], [m1,m2], [p1,p2], BindingMode.PRODUCT)` it results in the following executables:  
c1-p1-m1, c1-p1-m2, c1-p2-m1, c1-p2-m2, c2-p1-m1, c2-p1-m2, c2-p2-m1, c2-p2-m2.

In [ ]:
binding_product = CircuitBinding(
    [c1, c2], values=[v1], measurements=[m1, m1_bis], mode=BindingMode.PRODUCT
)

print(run(binding_product, AWSDevice.BRAKET_LOCAL_SIMULATOR))

0


BatchResult: 4 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 1.0
      Error/Variance: None
    With observables: [pI]
    

    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 0.5158102766798419
      Error/Variance: None
    With observables: [pZ]
    

    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 1.0
      Error/Variance: None
    With observables: [pI]
    

    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 0.046442687747035576
      Error/Variance: None
    With observables: [pZ]
    



#### BindingMode.ZIP
This binding mode has more subtleties than the product mode, it is split between 2 cases: if the number of Circuits (or CircuitBinding) is either 1 or n which is zipped with a n number of parameters and measurements. It's akin to the zip method between two lists.
If we have `CircuitBinding([c1,c2], [p1,p2], [m1,m2], BindingMode.ZIP)` it results in the following executables: c1-p1-m1, c2-p2-m2.
If we have 1 circuit then it is as follow: `CircuitBinding([c1], [p1,p2], [m1,m2], BindingMode.ZIP)` we have: c1-p1-m1, c1-p2-m2.

Note: if the length of the list circuits is different of the length of the lists measurements and values then it will raise an Error, we can only have the lengths (1,n,n) or (n,n,n)

In [ ]:
binding_zip = CircuitBinding(
    [c1, c2], values=[v1, v2], measurements=[m1, m1_bis], mode=BindingMode.ZIP
)

print(run(binding_zip, AWSDevice.BRAKET_LOCAL_SIMULATOR))

0
BatchResult: 2 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 1.0
      Error/Variance: None
    With observables: [pI]
    

    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [2.0], 'phi': [2.0], 'psi': [2.0]}
      Expectation value: -0.008893280632411068
      Error/Variance: None
    With observables: [pZ]
    



In [ ]:
binding_zip_1_circuit = CircuitBinding(
    [CircuitBinding([c1, c2])],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.ZIP,
)

print(run(binding_zip_1_circuit, AWSDevice.BRAKET_LOCAL_SIMULATOR))

0
BatchResult: 2 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 1.0
      Error/Variance: None
    With observables: [pI]
    

    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 1.0
      Error/Variance: None
    With observables: [pI]
    



Note: There is an edge case if the circuit list only contains 1 CircuitBinding then it'll act like it's one circuit but iterated through every circuits in the embedded CircuitBinding.

In [ ]:
binding_zip_1_cb = CircuitBinding([c1], [v1, v2], [m1, m1_bis], mode=BindingMode.ZIP)

print(run(binding_zip_1_cb, AWSDevice.BRAKET_LOCAL_SIMULATOR))

-1
BatchResult: 2 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 1.0
      Error/Variance: None
    With observables: [pI]
    

    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [2.0], 'phi': [2.0], 'psi': [2.0]}
      Expectation value: -0.43478260869565216
      Error/Variance: None
    With observables: [pZ]
    



### Usecase 3: Embedded CircuitBindings.

In the case of more complex desired binding it is possible to have a CircuitBinding inside another CircuitBinding (in the circuits list). In this case the checks in the ZIP mode consider a circuitBinding as 1 circuit.  
In the case we bind a measurement to an embedded circuitbinding it outputs:  
`CircuitBinding([CircuitBinding([c1,c2], [p1,p2], PRODUCT), c3], [m1,m2], ZIP)` => c1-p1-m1, c1-p2-m1, c2-p1-m1, c2-p2-m1, c3-m2.

As we see if one of the parameter is not inputted in the embedded circuitbinding it merges the outer parameters with the inner ones. However if both are reference it will proceed like this:  
- inner parameters-measurement bind  
- inner parameters-outer measurements  
- outer parameters-inner measurements  
- outer parameters-outer measurements  

Example: `CircuitBinding([CircuitBinding(c1, p1, m1), c2], [p2], m2, PRODUCT)`  
c1-p1-m1, c1-p2-m1, c1-p1-m2, c1-p2-m2, c2-p2-m2

In [ ]:
binding_embedded = CircuitBinding(
    [CircuitBinding([c1], values=[v1], measurements=m1), c2],
    values=[v2],
    measurements=[m1_bis],
    mode=BindingMode.PRODUCT,
)

print(run(binding_embedded, AWSDevice.BRAKET_LOCAL_SIMULATOR))

0
BatchResult: 5 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 1.0
      Error/Variance: None
    With observables: [pI]
    

    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
      Expectation value: 0.5523715415019763
      Error/Variance: None
    With observables: [pZ]
    

    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [2.0], 'phi': [2.0], 'psi': [2.0]}
      Expectation value: 1.0
      Error/Variance: None
    With observables: [pI]
    

    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [2.0], 'phi': [2.0], 'psi': [2.0]}
      Expectation value: -0.4051383399209486
      Error/Variance: None
    With observables: [pZ]
    

    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [2.0], 'phi': [2.0], 'psi': [2.0]}
      Expectation 